In [3]:
from sklearn.ensemble import BaggingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

df = pd.read_csv('cyberbullying_datasets/course_dataset/cb_chi_compressed.csv.gz')


X = df.drop(columns=["bullying"])
y = df["bullying"]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

bagged_svm = BaggingClassifier(
    estimator=SVC(kernel="rbf", C=1.0,gamma="scale", random_state=0),
    n_estimators=70,
    random_state=0
).fit(X_train, y_train)

y_pred = bagged_svm.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.7660343270099368

Classification Report:
               precision    recall  f1-score   support

           0       0.73      0.85      0.78      1105
           1       0.82      0.68      0.75      1109

    accuracy                           0.77      2214
   macro avg       0.77      0.77      0.76      2214
weighted avg       0.77      0.77      0.76      2214



### Extract features and justify the methods used

In [8]:
import pandas as pd

df = pd.read_csv('cyberbullying_datasets/course_dataset/clean_cb.csv')
print("✅ Data successfully loaded!")
print(f"Number of rows: {len(df)}")
print("\nFirst few rows:")
print(df.head())

✅ Data successfully loaded!
Number of rows: 11100

First few rows:
                                                Text  CB_Label  \
0  damn there is someones nana up here at beach w...         0   
1  no kidding! dick clark was a corpse mechanical...         0   
2  i read an article on jobros and thought damn w...         0   
3  I got one fucking day of sprinkles and now it'...         0   
4  I was already listening to Elliott smith  and ...         0   

                                          clean_text  
0  damn someone nana beach not think ic steal qui...  
1  kidding ! dick clark corpse mechanically opera...  
2  read article jobros think damn cash jobro poke...  
3    get fucking day sprinkle sunshine douchebaggery  
4  listen elliott smith fucking hate kanye west v...  


In [13]:
print(df['clean_text'].isna().sum())
df = df.dropna(subset=['clean_text'])
print(df['clean_text'].isna().sum())


31
0


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=2000,        # ta med de 2000 mest relevante ordene
    ngram_range=(1,2),        # inkluderer 1-gram (enkeltord) og 2-gram (to ord sammen)
    # min_df=5,                 # ord må forekomme i minst 5 dokumenter
    # stop_words='english'      # fjerner vanlige engelske ord
)

X_tfidf = tfidf.fit_transform(df['clean_text'])


In [1]:
# import matplotlib.pyplot as plt
# from wordcloud import WordCloud

# # Lag en stor tekst-streng av alle clean_text
# all_text = ' '.join(df['clean_text'].dropna())

# # Lag WordCloud
# wordcloud = WordCloud(width=800, height=400, background_color='white', max_words=100).generate(all_text)

# # Visualiser
# plt.figure(figsize=(15,7))
# plt.imshow(wordcloud, interpolation='bilinear')
# plt.axis('off')
# plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X_tfidf = tfidf.fit_transform(df['clean_text'])

# Henter ut alle dokumenter (i vårt tilfelle bare én)
tfidf_sum = np.array(X_tfidf.sum(axis=0)).flatten()

# Hent ordnavn
words = tfidf.get_feature_names_out()

# Lag en DataFrame
tfidf_df = pd.DataFrame({'word': words, 'tfidf': tfidf_sum})

# Topp 20 ord
top_words = tfidf_df.sort_values(by='tfidf', ascending=False).head(20)

#Topp 20 bigrams
bigrams_df = tfidf_df[tfidf_df['word'].str.contains(' ')]
top20_bigrams = bigrams_df.sort_values(by='tfidf', ascending=False).head(20)


# # Barplot for top 20 ord
# plt.figure(figsize=(14,8))
# plt.barh(top_words['word'][::-1], top_words['tfidf'][::-1], color='purple')
# plt.xlabel('TF-IDF score')
# plt.title('Top 20 words by TF-IDF')
# plt.show()

# # Barplot for topp 20 bigrams
# plt.figure(figsize=(12,8))
# plt.barh(top20_bigrams["word"][::-1], top20_bigrams['tfidf'][::-1], color='green')
# plt.xlabel("TF-IDF score")
# plt.title("Top 20 bigrams by TF-IDF")
# plt.show()



In [ ]:
for label in [0,1]:
    texts = df[df['CB_Label']==label]['clean_text']
    X_label = tfidf.fit_transform(texts)
    feature_names = tfidf.get_feature_names_out()
    tfidf_sum = np.array(X_label.sum(axis=0)).flatten()
    tfidf_df = pd.DataFrame({'word': feature_names, 'tfidf': tfidf_sum})
    top_words = tfidf_df.sort_values(by='tfidf', ascending=False).head(20)

    
    plt.figure(figsize=(12,8))
    plt.barh(top_words['word'][::-1], top_words['tfidf'][::-1])
    plt.title(f'Top 20 unigrams for CB_Label={label}')
    plt.xlabel('TF-IDF score')
    plt.show()


In [ ]:
for label in [0,1]:
    texts = df[df['CB_Label']==label]['clean_text']
    X_label = tfidf.fit_transform(texts)
    feature_names = tfidf.get_feature_names_out()
    tfidf_sum = np.array(X_label.sum(axis=0)).flatten()
    tfidf_df = pd.DataFrame({'ngram': feature_names, 'tfidf': tfidf_sum})
    bigrams_df = tfidf_df[tfidf_df['ngram'].str.contains(' ')]
    top10 = bigrams_df.sort_values(by='tfidf', ascending=False).head(20)
    
    plt.figure(figsize=(12,8))
    plt.barh(top10['ngram'][::-1], top10['tfidf'][::-1])
    plt.title(f'Top 20 bigrams for CB_Label={label}')
    plt.xlabel('TF-IDF score')
    plt.show()
